# Victorian spatial demand and weather map

This notebook builds the interactive map used to choose weather locations for the Victorian electricity-demand forecast. It is written for readers without electricity-market or mapping expertise.

It answers three questions:

1. Where are the largest publicly reported terminal-station demand peaks?
2. Which seven terminal locations are selected as demand-related weather coordinates?
3. How do those locations compare with the two Melbourne weather locations used by AEMO forecasting?

## Install the required libraries

Run this cell once for each new notebook environment. `%pip` installs the libraries into the active Jupyter kernel.

In [1]:
%pip install --quiet "folium==0.20.0" "openpyxl>=3.1,<4" "pandas>=2.2,<3" "requests>=2.32,<3"

Note: you may need to restart the kernel to use updated packages.


## 1. Data and interpretation

The map combines two public sources:

- **Victorian terminal-station demand workbook:** summer and winter maximum demand published in megawatts (MW). MW means the amount of electricity needed at one moment.
- **VicGrid spatial data:** coordinates for terminal stations, transmission lines and the Victorian boundary.

A terminal station is a major network delivery point. Public terminal-demand data do not reveal the addresses of the customers using the electricity. We therefore make a testable modelling assumption: a terminal with high demand may indicate substantial electricity use in the surrounding area, so weather at that coordinate may help predict Victorian demand. The map supports this hypothesis; it does not prove it.

The summer and winter values are individual maxima. Different terminals may reach their maximum at different times, so the values can be compared but must not be added together.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import IFrame

# Find this notebook package whether Jupyter starts here or above the repository.
working_folder = Path.cwd().resolve()
search_roots = [working_folder, *working_folder.parents]
notebook_candidates = [working_folder]
for root in search_roots:
    notebook_candidates.extend(
        [
            root / "notebooks" / "spatial_demand_map",
            root / "Electricity_Forecast_Databricks_MLOps" / "notebooks" / "spatial_demand_map",
        ]
    )

notebook_folder = next(
    folder
    for folder in notebook_candidates
    if (folder / "config.json").exists() and (folder / "build_map.py").exists()
)
sys.path.insert(0, str(notebook_folder))

import build_map as map_builder

config = map_builder.load_config(notebook_folder / "config.json")

## 2. Load the public source files

The source files are downloaded once and retained in the existing cache. Reusing the cache makes repeated notebook runs fast and keeps the exact input files available for audit.

In [3]:
source_files = map_builder.fetch_sources(config, refresh=False)
terminal_coordinates = map_builder.load_geojson(source_files["terminal_stations"])

pd.DataFrame(
    {
        "Source": source_files.keys(),
        "Cached file": [path.name for path in source_files.values()],
    }
)

,Source,Cached file
0,terminal_demand,terminal_station_demand_forecasts_2025.xlsx
1,terminal_stations,vicgrid_terminal_stations.geojson
2,transmission_lines,vicgrid_transmission_lines.geojson
3,victoria_boundary,victoria_boundary.geojson


## 3. Reduce the workbook to 31 comparable terminal locations

The workbook contains more rows than locations because a terminal can contain several voltage buses and can also appear under overlapping location definitions. The map applies the following rules:

1. Read the published summer-maximum and winter-maximum sheets.
2. Use the actual value for the stated year and keep values measured in MW.
3. Keep rows whose location type is **Entire**. Exclude **Hybrid** and **Split Bus** because combining them with Entire rows could count the same demand twice.
4. Add voltage-bus rows that belong to the same terminal.
5. Standardise the few station-code aliases and join each terminal to its VicGrid coordinate.

After these rules, both seasonal layers contain the same **31 matched terminal locations**.

In [4]:
demand_settings = config["sources"]["terminal_demand"]
points_by_measure = {}
filter_summary = []

for measure in demand_settings["measures"]:
    workbook_rows = pd.read_excel(
        source_files["terminal_demand"],
        sheet_name=measure["sheet"],
        header=demand_settings["header_row"],
    )
    terminal_demand, audit = map_builder.prepare_terminal_demand(
        workbook_rows,
        year=measure["year"],
        unit=demand_settings["unit"],
        included_location_types=demand_settings["included_location_types"],
        aliases=config["station_code_aliases"],
    )
    mapped_points, unmatched_codes = map_builder.join_demand_to_terminal_stations(
        terminal_demand, terminal_coordinates
    )
    points_by_measure[measure["id"]] = mapped_points
    filter_summary.append(
        {
            "Layer": measure["label"],
            "Workbook rows": audit["raw_rows"],
            "Entire voltage rows": audit["selected_voltage_rows"],
            "Mapped terminals": len(mapped_points),
            "Unmatched codes": len(unmatched_codes),
        }
    )

pd.DataFrame(filter_summary)

,Layer,Workbook rows,Entire voltage rows,Mapped terminals,Unmatched codes
0,Summer 2025 maximum,161,39,31,0
1,Winter 2024 maximum,161,39,31,0


## 4. Select the seven highest summer-demand locations

The default summer layer is ranked from highest to lowest maximum demand. The first seven become the red-star demand locations. The same seven remain selected when the map switches to winter so that changing the displayed season does not silently change the experiment.

This ranking identifies candidate coordinates; it does not prove that electricity is consumed at the terminal or that weather there will improve the regional forecast.

In [5]:
default_measure = next(
    measure for measure in demand_settings["measures"] if measure["default"]
)
summer_points = points_by_measure[default_measure["id"]]
selected_codes = set(
    summer_points.nlargest(config["map"]["selected_driver_count"], "demand_mw")[
        "station_code"
    ]
)

ranked_points = map_builder.combine_measure_points(
    points_by_measure, selected_codes
).sort_values("summer_maximum_mw", ascending=False)
ranked_points["Summer rank"] = range(1, len(ranked_points) + 1)
ranked_points["Winter rank"] = (
    ranked_points["winter_maximum_mw"].rank(ascending=False).astype(int)
)

top_seven = ranked_points.loc[
    ranked_points["selected_demand_driver"],
    [
        "Summer rank",
        "station_name",
        "summer_maximum_mw",
        "Winter rank",
        "winter_maximum_mw",
    ],
].rename(
    columns={
        "station_name": "Location",
        "summer_maximum_mw": "Summer maximum (MW)",
        "winter_maximum_mw": "Winter maximum (MW)",
    }
)
top_seven["Selected"] = "Yes"
top_seven.round(1)

,Summer rank,Location,Summer maximum (MW),Winter rank,Winter maximum (MW),Selected
13,1,Keilor,568.2,1,456.9,Yes
19,2,Rowville,534.1,2,431.8,Yes
4,3,Cranbourne,524.5,4,423.4,Yes
20,4,Richmond,473.7,6,400.3,Yes
27,5,Thomastown,472.5,5,408.3,Yes
5,6,East Rowville,458.1,3,425.1,Yes
8,7,Geelong,432.8,7,398.3,Yes


## 5. Add the AEMO forecasting locations and build the map

The two purple diamonds are Melbourne Olympic Park and Melbourne Airport. AEMO identifies them among the main weather stations used by its NEM Demand Forecasting System for Victoria. They form a second, evidence-backed location strategy to compare with the seven terminal-demand coordinates.

The final map uses the maintained renderer so its heat layer, circles, labels, controls and explanations stay aligned with the tested HTML map. Circle size and heat intensity use the same seasonal maximum MW value.

In [6]:
demand_map = map_builder.build_map(
    points_by_measure,
    measures=demand_settings["measures"],
    selected_driver_codes=selected_codes,
    transmission_geojson=map_builder.load_geojson(source_files["transmission_lines"]),
    boundary_geojson=map_builder.load_geojson(source_files["victoria_boundary"]),
    weather_locations=config["weather_locations"],
    map_config=config["map"],
    source_page=demand_settings["landing_page"],
)

html_path = notebook_folder / "victoria_terminal_demand_weather_map.html"
demand_map.save(html_path)
print(f"Saved map: {html_path}")

Saved map: /Users/franciscorealescastro/Documents/AEMO_VIC1_Databricks_MLOps/Electricity_Forecast_Databricks_MLOps/notebooks/spatial_demand_map/victoria_terminal_demand_weather_map.html


## 6. View the completed interactive map

The HTML file is stored beside this notebook. Use the season buttons to switch between the actual summer and winter maxima, and use the view buttons to move between the selected demand area, the AEMO weather locations and all of Victoria.

In [7]:
IFrame(src=html_path.name, width="100%", height=750)